In [ ]:
# Move kaggle.json to the right place
import os
from google.colab import files # Import files module

os.makedirs("/root/.kaggle", exist_ok=True)

# Check if kaggle.json exists. If not, prompt user to upload.
if not os.path.exists("kaggle.json"):
    print("kaggle.json not found. Please upload your kaggle.json file.")
    files.upload() # This will prompt the user

# Now, attempt to move the file
# Check again after potential upload
if os.path.exists("kaggle.json"):
    os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 600)
else:
    print("kaggle.json still not found after attempting upload. Please ensure you upload the correct file.")

# Install kaggle library
!pip install kaggle -q

# Download the dataset
# This part might fail if kaggle.json was not successfully moved and configured.
!kaggle competitions download -c jigsaw-toxic-comment-classification-challenge

# Unzip it
!unzip -q jigsaw-toxic-comment-classification-challenge.zip -d data/
!ls data/

In [ ]:
import zipfile, os

# Unzip all zip files inside the data folder
for file in os.listdir("data"):
    if file.endswith(".zip"):
        with zipfile.ZipFile(f"data/{file}", "r") as z:
            z.extractall("data/")
            print(f"Extracted: {file}")

print("\nData folder now:")
print(os.listdir("data"))

In [ ]:
import pandas as pd

df = pd.read_csv("data/train.csv")

print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
print("\nLabel counts:")
print(df[label_cols].sum())
print("\nLabel percentages:")
print((df[label_cols].mean() * 100).round(2))

In [ ]:
import pandas as pd

df = pd.read_csv("data/train.csv")

print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
print("\nLabel counts:")
print(df[label_cols].sum())
print("\nLabel percentages:")
print((df[label_cols].mean() * 100).round(2))

print("\nAny missing values?")
print(df.isnull().sum())

In [ ]:
!pip install emoji indic-nlp-library -q

In [ ]:
import re
import emoji

def clean_text(text):
    text = re.sub(r"http\S+|www\S+", "", text)   # remove URLs
    text = re.sub(r"<.*?>", "", text)              # remove HTML tags
    text = re.sub(r"\s+", " ", text).strip()       # fix whitespace
    return text

def handle_emojis(text):
    return emoji.demojize(text, delimiters=(" ", " "))

def normalize_repeated_chars(text):
    return re.sub(r"(.)\1{2,}", r"\1\1", text)    # soooo -> so

def normalize_abbreviations(text):
    abbrevs = {
        "wtf": "what the hell", "stfu": "shut up",
        "idk": "i don't know", "ngl": "not gonna lie",
        "lmao": "laughing", "omg": "oh my god",
        "u": "you", "ur": "your", "r": "are",
        "bc": "because", "2": "to", "4": "for"
    }
    tokens = text.lower().split()
    tokens = [abbrevs.get(t, t) for t in tokens]
    return " ".join(tokens)

def full_pipeline(text):
    text = clean_text(text)
    text = handle_emojis(text)
    text = normalize_repeated_chars(text)
    text = normalize_abbreviations(text)
    return text.lower().strip()

# Test it on a few samples
test_samples = [
    "Check this out http://example.com 😡😡 ur sooooo stupid wtf",
    "yaar kya scene hai bro 🔥🔥🔥",
    "stfu u idiot lmaooo",
]

for s in test_samples:
    print("BEFORE:", s)
    print("AFTER: ", full_pipeline(s))
    print()

In [ ]:
from multiprocessing import Pool, cpu_count
from tqdm import tqdm

print(f"CPU cores available: {cpu_count()}")

with Pool(cpu_count()) as pool:
    clean_texts = list(tqdm(
        pool.imap(full_pipeline, df["comment_text"].tolist()),
        total=len(df),
        desc="Preprocessing"
    ))

df["clean_text"] = clean_texts

print(df[["comment_text", "clean_text"]].head(10))
print("\nDone! Total rows processed:", len(df))

In [ ]:
from sklearn.model_selection import train_test_split

# Create a binary label — 1 if ANY label is toxic, 0 if clean
df["label"] = (df[["toxic", "severe_toxic", "obscene",
                    "threat", "insult", "identity_hate"]].sum(axis=1) > 0).astype(int)

print("Label distribution:")
print(df["label"].value_counts())
print("\nPercentage:")
print((df["label"].value_counts(normalize=True) * 100).round(2))

# Split
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["label"])

print(f"\nTrain size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")

In [ ]:
!pip install transformers torch -q

import torch
from transformers import AutoTokenizer

MODEL_NAME = "google/muril-base-cased"

print("Loading MuRIL tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Test the tokenizer
sample = "yaar tu bahut bura hai bro"
tokens = tokenizer(sample, return_tensors="pt")
print("\nSample text:", sample)
print("Token IDs:", tokens["input_ids"])
print("Tokens:", tokenizer.convert_ids_to_tokens(tokens["input_ids"][0]))
print("\nTokenizer working!")

In [ ]:
from torch.utils.data import Dataset

class AbuseDataset(Dataset):
    def __init__(self, texts, labels, max_len=128):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.float)
        }

# Create datasets
train_dataset = AbuseDataset(
    train_df["clean_text"].tolist(),
    train_df["label"].tolist()
)
val_dataset = AbuseDataset(
    val_df["clean_text"].tolist(),
    val_df["label"].tolist()
)

print("Train dataset size:", len(train_dataset))
print("Val dataset size:  ", len(val_dataset))
print("\nSample item keys:", train_dataset[0].keys())

In [ ]:
from transformers import AutoModelForSequenceClassification

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1  # binary classification
)
model = model.to(DEVICE)

print("Model loaded!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
import os
from google.colab import userdata
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
from tqdm import tqdm
import wandb
import torch

# ---- W&B Setup via Colab Secrets ----
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

wandb.init(
    project="multilingual-abuse-detection",
    name="muril-binary-v1",
    config={
        "model":      MODEL_NAME,
        "batch_size": 32,       # increased from 16 — faster on GPU
        "epochs":     3,
        "lr":         2e-5,
        "max_len":    128,
        "optimizer":  "AdamW",
        "dataset":    "kaggle-toxic-comments"
    }
)

# Config
BATCH_SIZE = 32       # bumped up for GPU efficiency
EPOCHS     = 3
LR         = 2e-5

# Dataloaders — pin_memory + num_workers speeds up data loading
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          pin_memory=True, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          pin_memory=True, num_workers=2)

# Class weights to handle imbalance
pos = train_df["label"].sum()
neg = len(train_df) - pos
pos_weight = torch.tensor([neg / pos]).to(DEVICE)
print(f"Class weight (positive): {pos_weight.item():.2f}")

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=500,
    num_training_steps=total_steps
)

# ---- Mixed Precision Scaler (2-3x faster on T4 GPU) ----
scaler = torch.cuda.amp.GradScaler()

# Training loop
best_f1 = 0

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    total_loss = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=True)

    for i, batch in enumerate(train_bar):
        input_ids      = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels         = batch["labels"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad()

        # Mixed precision forward pass
        with torch.cuda.amp.autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits.squeeze(), labels)

        # Scaled backward pass
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

        # Update tqdm bar with live loss
        train_bar.set_postfix({"loss": f"{loss.item():.4f}"})

        if i % 100 == 0:
            wandb.log({"step_loss": loss.item(), "step": epoch * len(train_loader) + i})

    avg_loss = total_loss / len(train_loader)

    # --- Validate ---
    model.eval()
    all_preds, all_labels = [], []

    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]  ", leave=True)

    with torch.no_grad():
        for batch in val_bar:
            input_ids      = batch["input_ids"].to(DEVICE, non_blocking=True)
            attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
            labels         = batch["labels"].to(DEVICE, non_blocking=True)

            with torch.cuda.amp.autocast():
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            probs = torch.sigmoid(outputs.logits.squeeze())
            preds = (probs > 0.5).int()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.int().cpu().numpy())

            val_bar.set_postfix({"samples": len(all_preds)})

    val_f1 = f1_score(all_labels, all_preds, average="macro")

    wandb.log({
        "epoch":      epoch + 1,
        "train_loss": avg_loss,
        "val_f1":     val_f1,
    })

    print(f"\n✅ Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f}\n")

    if val_f1 > best_f1:
        best_f1 = val_f1
        model.save_pretrained("/content/best_model")
        tokenizer.save_pretrained("/content/best_model")
        print(f"  💾 Best model saved! (F1: {best_f1:.4f})\n")

wandb.finish()
print("Training complete! Best F1:", best_f1)